# boolean-mask-combine — worked example 3: In-range OR specially-marked, excluding masked-out

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `boolean-mask-combine`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Sometimes the keep-set is a union of conditions narrowed by an exclusion: keep entries that are either in a numeric range OR explicitly whitelisted, but always drop entries that are masked out. This needs `|` for the union, `&` to apply the exclusion, and `~` to negate the masked-out flag. Operator precedence means each comparison must be parenthesised.

## Worked solution

**Goal:** given a `(N,)` tensor `x`, a range `[lo, hi]`, a `(N,)` bool `whitelist`, and a `(N,)` bool `masked_out`, return a `(N,)` bool keep-mask that is `True` where (`lo <= x <= hi` OR `whitelist`) AND NOT `masked_out`.

**Step 1 — in-range predicate.** `(x >= lo) & (x <= hi)` gives a `(N,)` bool tensor that is `True` for values within the closed interval. Both comparisons are parenthesised so `&` does not swallow the bounds.

**Step 2 — union with the whitelist.** `in_range | whitelist`. `|` is elementwise OR, so an entry survives this stage if it is in range OR explicitly whitelisted — a union of two sets.

**Step 3 — apply the exclusion.** Masked-out entries must be dropped regardless of the above, so we AND with `~masked_out`. `~` negates the drop-flag into a keep-flag, and `&` intersects it with the union from step 2.

**Step 4 — combine.** `((x >= lo) & (x <= hi) | whitelist) & (~masked_out)`. Because `&` binds tighter than `|`, the in-range AND is evaluated before the OR — exactly the grouping we want — and the final `& (~masked_out)` applies to the whole union. To be unambiguous we still parenthesise generously.

**Why it works:** the final set is (range ∪ whitelist) ∩ (not masked-out). Union maps to `|`, intersection to `&`, negation to `~`; all operate elementwise on bool tensors of the same `(N,)` shape.

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat
np.random.seed(0)
t.manual_seed(0)

def keep_in_range_or_special(x: Tensor, lo: float, hi: float, whitelist: Tensor, masked_out: Tensor) -> Tensor:
    in_range = (x >= lo) & (x <= hi)
    return (in_range | whitelist) & (~masked_out)

x = t.tensor([0.0, 5.0, 9.0, 100.0, 50.0])
whitelist = t.tensor([False, False, False, True, False])
masked_out = t.tensor([False, False, False, False, True])
mask = keep_in_range_or_special(x, 1.0, 10.0, whitelist, masked_out)
print(mask)
print(mask.dtype, mask.shape)